In [ ]:
import numpy as np
from typing import Tuple, List, Optional
from IPython.display import display
from sympy import Matrix
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(linewidth=100000)

In [7]:
def output_matrix(X: np.ndarray, precision: int = 4):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    """
    X = np.array(X)
    if np.iscomplexobj(X):
        # Nếu ma trận chứa số phức
        display(Matrix(np.round(X.real, precision) + 1j * np.round(X.imag, precision)))
    else:
        # Ép kiểu về float, và làm tròn
        X_float = np.array(X, dtype=float)
        display(Matrix(np.round(X_float, precision)))

# Phương pháp Danielevsky tìm trị riêng và vectơ riêng


# Thuật toán
**Input**: $A_{n \times n}$.

**Output**: Đa thức đặc trưng $P(\lambda)$, các trị riêng $\lambda_i$, các vector riêng $v_i$.

1. Khởi tạo $P = A$, danh sách $L$ rỗng, $k = n$.

2. Lặp while $k \ge 2$:

    2.1. Xét $p_{k, k-1}$ của $P$:
    
    - Nếu $p_{k, k-1} \neq 0 \Rightarrow$ Qua bước 2.2.

    - Trái lại, tìm $j < k-1$ sao cho $p_{k, j} \neq 0$:

        - Nếu $\exists j \Rightarrow$ Tạo $C=I_{j \leftrightarrow k-1}$ (đổi chỗ hàng $j$ và hàng $k-1$ của $I$). Gán $P = C \cdot P \cdot C$. Thêm $C$ vào $L$, qua bước 2.2.

        - Trái lại:

            - Lặp $c$ từ $k$ đến $n-1$: 

                - Khởi tạo $X = [0]_{n \times n}$, gán $x_{i, c+1} = p_{i, c}\ \ \forall i < k$. 

                - Đặt $U = I + X$, $U^{-1} = I - X$.

                - Gán $P = U^{-1} \cdot P \cdot U$. Thêm $U$ vào $L$.

            - Nếu $p_{i, n} = 0 \ \ \forall i < k \Rightarrow$ Gán $k = k - 1$, quay lại đầu bước 2.
            
            - Trái lại, tạo $C=I_{n \leftrightarrow k-1}$. Gán $P = C \cdot P \cdot C$. Thêm $C$ vào $L$. Gán $k = n$, quay lại đầu bước 2.

    2.2. Khởi tạo $M = I$ và $S = I$. Thực hiện lần lượt các phép gán:

    - $m_{k-1, j} = p_{k, j} \quad \forall j \in [1, n]$.
    
    - $s_{k-1, j} = \dfrac{-p_{k, j}}{p_{k, k-1}}$ (với $j \neq k-1$) và $s_{k-1, k-1} = \dfrac{1}{p_{k, k-1}}$.

    - Gán $P = M \cdot P \cdot S$. Thêm $S$ vào $L$.
    
    - $k = k - 1$.

3. Sau khi lặp, thu được $P = \begin{bmatrix} F_1 & & \\ & F_2 & \\ & & \ddots \\ & & & F_s \end{bmatrix}$. Đặt $P(\lambda) = \displaystyle\prod\limits_{c=1}^s \det(F_c - \lambda I)$.

4. Lặp với từng khối $F_c \in \mathbb{R}^{m_c \times m_c}$ (nằm từ hàng $r_c$ đến $r_c+m_c-1$ trong $P$):

    4.1. Giải phương trình $\det(F_c - \lambda I) = 0$ thu được các $\lambda_i$.

    4.2. Lặp với từng $\lambda_i$:

    - Khởi tạo $u_i = [0]_{n \times 1}$.

    - Gán $u_i[r_c : r_c+m_c-1] = \begin{bmatrix} \lambda_i^{m_c-1} & \dots & \lambda_i & 1 \end{bmatrix}^T$

    - Tính và in ra $v_i = \left(\displaystyle\prod\limits_{Q \in L} Q \right) u_i$


In [ ]:
def danilevsky_method(A: np.ndarray, tol: float = 1e-10) -> Tuple[np.ndarray, List[np.ndarray]]:
    n = len(A)
    if A.shape != (n, n):
        raise ValueError("Ma trận đầu vào phải là ma trận vuông")
    
    P = A.copy()
    S_list = []
    
    k = n - 1
    while k >= 1:
        print(f"\n{'='*40}")
        print(f"BƯỚC k = {k+1}:")
        print(f"Ma trận hiện tại P:")
        output_matrix(P)
        
        if abs(P[k,k-1]) >= tol:
            print(f"TH1: p_{k+1},{k} ≠ 0")
            
            M = np.eye(n)
            M[k-1, :] = P[k, :]
            
            S = np.eye(n)
            S[k-1] = -P[k] / P[k,k-1]
            S[k-1,k-1] = 1.0 / P[k,k-1]
            
            print(f"Ma trận biến đổi M:")
            output_matrix(M)
            print(f"Ma trận biến đổi S = M^{{-1}}:")
            output_matrix(S)
            
            S_list.append(S)
            P = M @ P @ S
            k -= 1
            
        else:
            non_zero_idx = None
            for j in range(k-1, -1, -1):
                if abs(P[k,j]) >= tol:
                    non_zero_idx = j
                    break
            
            if non_zero_idx is not None:
                print(f"TH2: p_{k+1},{k} = 0, p_{k+1},{non_zero_idx+1} ≠ 0")
                
                C = np.eye(n)
                C[k-1,k-1] = 0
                C[non_zero_idx,non_zero_idx] = 0
                C[k-1,non_zero_idx] = 1
                C[non_zero_idx,k-1] = 1
                
                print(f"Ma trận hoán vị C (đổi chỗ hàng {k} và {non_zero_idx+1}):")
                output_matrix(C)
                
                P = C @ P @ C
                S_list.append(C)
                
                # Thực hiện biến đổi TH1 ngay sau khi hoán vị
                M = np.eye(n)
                M[k-1, :] = P[k, :]
                
                S = np.eye(n)
                S[k-1] = -P[k] / P[k,k-1]
                S[k-1,k-1] = 1.0 / P[k,k-1]
                
                print(f"Ma trận biến đổi M (sau hoán vị):")
                output_matrix(M)
                print(f"Ma trận biến đổi S (sau hoán vị):")
                output_matrix(S)
                
                S_list.append(S)
                P = M @ P @ S
                k -= 1
                
            else:
                print(f"TH3: p_{k+1},j = 0 với mọi j ≤ {k}")
                print("Tiến hành khử khối B nằm trên khối Frobenius:")
                for c in range(k, n - 1):
                    X = np.zeros((n, n))
                    X[:k, c+1] = P[:k, c]
                    U = np.eye(n) + X
                    U_inv = np.eye(n) - X
                    P = U_inv @ P @ U
                    S_list.append(U)
                    print(f"Ma trận biến đổi U:")
                    output_matrix(U)
                
                if np.all(np.abs(P[:k, n-1]) < tol):
                    print(f"Các phần tử p_{{i,n}} = 0 với mọi i ≤ {k}. Ma trận đã tách khối Frobenius.")
                    k -= 1
                    continue
                else:
                    print(f"Tồn tại p_{{i,n}} ≠ 0. Thực hiện hoán vị hàng {n} và {k}.")
                    C_n = np.eye(n)
                    C_n[k-1, k-1] = 0
                    C_n[n-1, n-1] = 0
                    C_n[k-1, n-1] = 1
                    C_n[n-1, k-1] = 1
                    print(f"Ma trận hoán vị C:")
                    output_matrix(C_n)
                    P = C_n @ P @ C_n
                    S_list.append(C_n)
                    k = n - 1
                    continue
    
    print(f"\n{'='*40}")
    print("Ma trận Frobenius cuối cùng P:")
    output_matrix(P)
    
    # --- TỰ ĐỘNG TÌM KHỐI FROBENIUS ---
    blocks = []
    start = 0
    for i in range(1, n):
        # Nếu phần tử dưới đường chéo chính không phải là 1 (hoặc xấp xỉ 1), đây là ranh giới cắt khối
        if abs(P[i, i-1] - 1.0) > tol:
            blocks.append((start, i))
            start = i
    blocks.append((start, n))
    
    print(f"Phát hiện {len(blocks)} khối Frobenius trên đường chéo.")
    
    all_eigenvalues = []
    eigenvectors = []
    poly_strs = []
    
    for start, end in blocks:
        mc = end - start
        rc = start
        print(f"-> Khối F_{len(poly_strs)+1} cấp {mc}x{mc} nằm từ hàng {rc+1} đến {rc+mc}.")
        
        coeffs = [1.0] + [-P[start, j] for j in range(start, end)]
        
        poly_str = f"(λ^{mc}"
        for i in range(1, mc+1):
            if abs(coeffs[i]) >= tol:
                sign = "+" if coeffs[i] > 0 else "-"
                # Loại bỏ in "λ^0"
                power_str = f"λ^{mc-i}" if mc-i > 0 else ""
                poly_str += f" {sign} {abs(coeffs[i]):.4f}{power_str}"
        poly_str += ")"
        poly_strs.append(poly_str)
        
        block_eigenvalues = np.roots(coeffs)
        all_eigenvalues.extend(block_eigenvalues)
        
        for lambda_i in block_eigenvalues:
            u_i = np.zeros(n, dtype=complex)
            core_v = np.array([lambda_i**(mc-1-i) for i in range(mc)])
            u_i[start:end] = core_v
            
            v = u_i
            for S in reversed(S_list):
                v = S @ v
            
            # Chỉ chuẩn hóa nếu norm > 0
            norm_v = np.linalg.norm(v)
            if norm_v > tol:
                v = v / norm_v
            eigenvectors.append(v)
            
    print("Phương trình đặc trưng P(λ) = 0:")
    print(" * ".join(poly_strs) + " = 0")
    
    return np.array(all_eigenvalues), eigenvectors

In [9]:


def read_matrix_from_file(filename: str) -> Optional[np.ndarray]:
    """
    Đọc ma trận từ file txt.
    
    Format file txt:
    - Mỗi hàng của ma trận trên một dòng
    - Các phần tử trong hàng cách nhau bởi dấu cách hoặc dấu phẩy
    
    Args:
        filename: Tên file txt chứa ma trận
        
    Returns:
        Ma trận numpy đọc được từ file
    """
    try:
        with open(filename, 'r') as f:
            lines = f.readlines()
            
        # Loại bỏ các dòng trống và khoảng trắng thừa
        lines = [line.strip() for line in lines if line.strip()]
        
        # Đọc ma trận
        matrix = []
        for line in lines:
            # Hỗ trợ cả dấu cách và dấu phẩy làm dấu phân cách
            row = [float(x.strip()) for x in line.replace(',', ' ').split()]
            matrix.append(row)
            
        return np.array(matrix)
    except FileNotFoundError:
        print(f"Không tìm thấy file {filename}")
        return None
    except Exception as e:
        print(f"Lỗi khi đọc file: {str(e)}")
        return None
    
# Ví dụ đọc ma trận từ file


In [10]:
try:
    # Đọc ma trận từ file
    A_file = read_matrix_from_file("DVSK_input_A.txt")
    if A_file is not None:
        print("Ma trận đọc từ file:")
        output_matrix(A_file)
        
        # Tìm trị riêng và vectơ riêng
        eigenvalues_file, eigenvectors_file = danilevsky_method(A_file)
        
        print("\nTrị riêng:")
        for i, lambda_i in enumerate(eigenvalues_file):
            if abs(lambda_i.imag) < 1e-10:
                print(f"λ_{i+1} = {lambda_i.real:.4f}")
            else:
                sign = "+" if lambda_i.imag > 0 else "-"
                print(f"λ_{i+1} = {lambda_i.real:.4f} {sign} {abs(lambda_i.imag):.4f}i")
                
        print("\nVectơ riêng:")
        for i, v in enumerate(eigenvectors_file):
            print(f"v_{i+1}:")
            # Loại bỏ phần ảo cực nhỏ do sai số máy tính
            v_clean = np.where(np.abs(v.imag) < 1e-10, v.real, v)
            output_matrix(v_clean)
except Exception as e:
    print(f"Lỗi khi xử lý ma trận: {str(e)}")

Ma trận đọc từ file:


Matrix([
[3.0,  5.0,  5.0, -2.0,  4.0],
[4.0,  1.0, -6.0,  0.0,  3.0],
[2.0,  0.0,  1.0,  7.0, -2.0],
[3.0, -1.0,  4.0,  5.0,  8.0],
[2.0,  4.0,  0.0,  0.0,  3.0]])


BƯỚC k = 5:
Ma trận hiện tại P:


Matrix([
[3.0,  5.0,  5.0, -2.0,  4.0],
[4.0,  1.0, -6.0,  0.0,  3.0],
[2.0,  0.0,  1.0,  7.0, -2.0],
[3.0, -1.0,  4.0,  5.0,  8.0],
[2.0,  4.0,  0.0,  0.0,  3.0]])

TH2: p_5,4 = 0, p_5,2 ≠ 0
Ma trận hoán vị C (đổi chỗ hàng 4 và 2):


...

Matrix([
[ 0.3022 - 0.4169*I],
[-0.5104 + 0.3954*I],
[ 0.1012 + 0.3913*I],
[-0.3177 - 0.1209*I],
[            0.1976]])

v_4:


Matrix([
[ 0.3022 + 0.4169*I],
[-0.5104 - 0.3954*I],
[ 0.1012 - 0.3913*I],
[-0.3177 + 0.1209*I],
[            0.1976]])

v_5:


Matrix([
[-0.7505],
[ 0.2848],
[-0.3543],
[ 0.2724],
[ 0.3949]])